# Text-to-SQL with Mistral models: measuring execution accuracy on BIRD

*Author: Masoud Rezvaninejad ([@smasoudrezvani](https://github.com/smasoudrezvani)), Independent*

Turning a question like *"What is the average weight of heroes as smart as Abomination?"* into SQL
looks easy, until the database has real-world quirks: a weight of `0` that means "unknown", numbers
stored as text, or several diagnoses packed into one field.

In this notebook you will:
1. Load real, messy databases from the [BIRD](https://bird-bench.github.io/) benchmark (mini-dev, 2026 release).
2. Build an **execution-accuracy scorer**: run the model's SQL and the correct SQL, and compare the results.
3. Try three ways of prompting a Mistral model: (a) the plain schema, (a) + **column notes**, and (c) an **execute-and-retry** loop.
4. See results for Ministral 3 8B, Mistral Small 4 and Mistral Large 4 on all 500 questions, and what *didn't* help.

The headline: adding short **column notes** (BIRD's column descriptions plus an automatic check for data traps) to the prompt raised Mistral Small 4 from **18.6% to 30.8%** on 500 questions, with no extra model calls.

All code, tests and per-question results are in the companion repo: https://github.com/smasoudrezvani/mistral-text2sql-cookbook

## Setup

We need the Mistral SDK and `remotezip`, which lets us download only the few databases we need from the 800 MB BIRD archive.

In [1]:
%pip install -q mistralai==3.7.0 remotezip==0.12.6

In [2]:
import os
from getpass import getpass

from mistralai.client import Mistral

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass("Mistral API key: ")
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

MODEL = "mistral-small-2603"  # also try "ministral-8b-2512" or "mistral-large-4" (slower: it reasons first)

Mistral API key: ··········


## 1. The data: BIRD mini-dev

BIRD has 500 hand-checked mini-dev questions over 11 real SQLite databases. Each record has a question, a short **hint** ("evidence") written by an expert, and the correct (**gold**) SQL. The questions come from Hugging Face; the databases from the official archive. To keep this notebook quick and cheap we use **30 fixed questions** from three databases (two of them full of messy values).

In [3]:
import json
import urllib.request
from pathlib import Path

from remotezip import RemoteZip

QUESTIONS_URL = "https://huggingface.co/datasets/birdsql/bird_mini_dev/resolve/main/data/mini_dev_sqlite.jsonl"
DB_ZIP_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip"
DEMO_DBS = ['california_schools', 'superhero', 'thrombosis_prediction']
DEMO_IDS = [10, 18, 24, 25, 35, 37, 339, 348, 350, 356, 361, 363, 368, 374, 375, 382, 387, 391, 404, 406, 408, 413, 414, 420, 427, 433, 438, 442, 444, 454]

with urllib.request.urlopen(QUESTIONS_URL) as f:
    questions = [json.loads(line) for line in f.read().decode("utf-8").splitlines() if line.strip()]
demo = [q for q in questions if q["question_id"] in DEMO_IDS]

DATA = Path("bird")
with RemoteZip(DB_ZIP_URL) as z:  # downloads only these databases (~19 MB), not the whole archive
    for name in z.namelist():
        if any(f"/dev_databases/{db}/" in name for db in DEMO_DBS) and not name.endswith("/"):
            z.extract(name, DATA)
DB_ROOT = DATA / "minidev/MINIDEV/dev_databases"


def db_file(db_id: str) -> Path:
    return DB_ROOT / db_id / f"{db_id}.sqlite"


print(len(questions), "questions in mini-dev;", len(demo), "in this demo")
print(json.dumps(demo[7], indent=1))

500 questions in mini-dev; 30 in this demo
{
 "question_id": 348,
 "db_id": "superhero",
 "question": "I want to profile the cohort of heroes whose Intelligence score matches Abomination's Intelligence score. What is their average measured weight in kg? Please round the average to two decimals.",
 "evidence": "Intelligence is one of the six attribute scores a hero has (Intelligence, Strength, Speed, Durability, Power, Combat)",
 "SQL": "SELECT ROUND(AVG(CASE WHEN s.weight_kg > 0 THEN s.weight_kg END), 2) AS average_weight_kg FROM superhero s JOIN hero_attribute ha ON ha.hero_id = s.id JOIN attribute a ON a.id = ha.attribute_id WHERE a.attribute_name = 'Intelligence' AND ha.attribute_value = (SELECT abom_ha.attribute_value FROM superhero abom JOIN hero_attribute abom_ha ON abom_ha.hero_id = abom.id JOIN attribute abom_a ON abom_a.id = abom_ha.attribute_id WHERE abom.superhero_name = 'Abomination' AND abom_a.attribute_name = 'Intelligence')",
 "difficulty": "challenging"
}


## 2. The scorer: execution accuracy

We don't compare SQL text (two different queries can both be right). We **run** the model's query and the gold query and compare the rows:

- **BIRD rule** (the official one): the two results are equal as *sets* (row order and duplicates ignored).
- **Strict rule**: duplicates count, and order counts when the gold query has an outer `ORDER BY`.

The database is opened **read-only** (a model's `DELETE` can't damage it), and a timeout really stops a runaway query.

In [4]:
# --- copied from src/text2sql_eval/scorer.py in the companion repo ---
import logging
import re
import sqlite3
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

logger = logging.getLogger(__name__)

Row = tuple[object, ...]
Mode = Literal["bird", "strict"]

DEFAULT_TIMEOUT_S = 30.0  # same as BIRD
DEFAULT_MAX_ROWS = 200_000  # guard against huge results on a small-RAM machine
_PROGRESS_EVERY_N_STEPS = 10_000


@dataclass(frozen=True)
class ExecResult:
    """Outcome of running one SQL query.

    Attributes:
        rows: Result rows, or None if the query failed or timed out.
        error: SQLite error message, or None.
        timed_out: True if the query was stopped by the timeout.
        seconds: Wall-clock run time.
    """

    rows: list[Row] | None
    error: str | None = None
    timed_out: bool = False
    seconds: float = 0.0

    @property
    def ok(self) -> bool:
        """True if the query ran to completion and returned rows (possibly zero)."""
        return self.rows is not None and self.error is None and not self.timed_out


@dataclass(frozen=True)
class Score:
    """Correctness of one prediction under both rules.

    Attributes:
        bird: Correct under BIRD's set rule.
        strict: Correct under the strict rule.
    """

    bird: bool
    strict: bool


def execute_sql(
    db_path: Path,
    sql: str,
    timeout_s: float = DEFAULT_TIMEOUT_S,
    max_rows: int | None = DEFAULT_MAX_ROWS,
    attach: list[Path] | None = None,
) -> ExecResult:
    """Run one SQL query on a SQLite database, read-only, with a timeout.

    Args:
        db_path: Path to the ``.sqlite`` file.
        sql: A single SQL statement.
        timeout_s: Stop the query after this many seconds.
        max_rows: Fail if the result has more rows than this (None = no limit).
        attach: Extra databases to ATTACH read-only (e.g. distractor tables). Unqualified
            table names resolve in the main database first.

    Returns:
        An ExecResult with rows, or with the error / timeout recorded.
    """
    start = time.monotonic()
    deadline = start + timeout_s
    stopped = False

    def _check_deadline() -> int:
        nonlocal stopped
        if time.monotonic() > deadline:
            stopped = True
            return 1  # non-zero tells SQLite to abort the query
        return 0

    try:
        con = sqlite3.connect(f"{Path(db_path).resolve().as_uri()}?mode=ro", uri=True)
    except sqlite3.Error as e:
        return ExecResult(rows=None, error=str(e), seconds=time.monotonic() - start)

    try:
        for i, extra in enumerate(attach or []):
            con.execute(
                f"ATTACH DATABASE ? AS extra{i}", (f"{Path(extra).resolve().as_uri()}?mode=ro",)
            )
        con.set_progress_handler(_check_deadline, _PROGRESS_EVERY_N_STEPS)
        cur = con.execute(sql)
        rows: list[Row] = []
        while batch := cur.fetchmany(1_000):
            rows.extend(batch)
            if max_rows is not None and len(rows) > max_rows:
                return ExecResult(
                    rows=None,
                    error=f"too many rows (> {max_rows})",
                    seconds=time.monotonic() - start,
                )
        return ExecResult(rows=rows, seconds=time.monotonic() - start)
    except (sqlite3.Error, sqlite3.Warning) as e:
        elapsed = time.monotonic() - start
        if stopped:
            logger.debug("Query timed out after %.1f s", elapsed)
            return ExecResult(rows=None, error="timeout", timed_out=True, seconds=elapsed)
        return ExecResult(rows=None, error=str(e), seconds=elapsed)
    finally:
        con.close()


def results_match(
    pred_rows: list[Row], gold_rows: list[Row], mode: Mode = "bird", ordered: bool = False
) -> bool:
    """Compare two result sets.

    Args:
        pred_rows: Rows returned by the predicted query.
        gold_rows: Rows returned by the gold query.
        mode: ``"bird"`` (set equality) or ``"strict"`` (multiset, or list if ordered).
        ordered: Whether row order matters (only used in strict mode).

    Returns:
        True if the results count as equal under the chosen rule.

    Raises:
        ValueError: If mode is unknown.
    """
    if mode == "bird":
        return set(pred_rows) == set(gold_rows)
    if mode == "strict":
        if ordered:
            return list(pred_rows) == list(gold_rows)
        return Counter(pred_rows) == Counter(gold_rows)
    raise ValueError(f"unknown mode: {mode!r}")


_ORDER_BY = re.compile(r"\bORDER\s+BY\b", re.IGNORECASE)


def has_outer_order_by(sql: str) -> bool:
    """Check whether the outermost query has ORDER BY.

    ORDER BY inside parentheses (subqueries, window functions), string literals, quoted
    identifiers and comments is ignored.

    Args:
        sql: A SQL statement.

    Returns:
        True if ORDER BY appears at parenthesis depth 0.
    """
    out: list[str] = []
    depth = 0
    i, n = 0, len(sql)
    closers = {"'": "'", '"': '"', "`": "`", "[": "]"}
    while i < n:
        ch = sql[i]
        if ch in closers:  # skip a string literal or quoted identifier
            end = sql.find(closers[ch], i + 1)
            i = n if end == -1 else end + 1
            out.append(" ")
            continue
        if sql.startswith("--", i):  # line comment
            end = sql.find("\n", i)
            i = n if end == -1 else end + 1
            out.append(" ")
            continue
        if sql.startswith("/*", i):  # block comment
            end = sql.find("*/", i + 2)
            i = n if end == -1 else end + 2
            out.append(" ")
            continue
        if ch == "(":
            depth += 1
            out.append(" ")
        elif ch == ")":
            depth = max(0, depth - 1)
            out.append(" ")
        else:
            out.append(ch if depth == 0 else " ")
        i += 1
    return bool(_ORDER_BY.search("".join(out)))


def compare(pred: ExecResult, gold: ExecResult, gold_sql: str) -> Score:
    """Score one prediction against an already-executed gold query.

    Args:
        pred: Result of running the predicted SQL.
        gold: Result of running the gold SQL (must have succeeded).
        gold_sql: The gold SQL text (used to decide whether order matters).

    Returns:
        Score under both the BIRD and the strict rule. A failed prediction is wrong.

    Raises:
        ValueError: If the gold query did not run successfully.
    """
    if not gold.ok:
        raise ValueError(f"gold query failed: {gold.error}")
    if not pred.ok:
        return Score(bird=False, strict=False)
    assert pred.rows is not None and gold.rows is not None
    ordered = has_outer_order_by(gold_sql)
    return Score(
        bird=results_match(pred.rows, gold.rows, mode="bird"),
        strict=results_match(pred.rows, gold.rows, mode="strict", ordered=ordered),
    )

In [5]:
GOLD = {q["question_id"]: execute_sql(db_file(q["db_id"]), q["SQL"], timeout_s=120) for q in demo}
assert all(g.ok for g in GOLD.values())
print("gold answers ready for", len(GOLD), "questions")

gold answers ready for 30 questions


## 3. The prompt: schema, example rows, hint, question

The model sees each table's `CREATE TABLE` statement and 3 example rows, then the hint and the question, and replies with one SQL query in a code block. Some Mistral models (like Large 4) reason before answering; we keep only the answer text.

In [6]:
# --- copied from src/text2sql_eval/schema.py in the companion repo ---
import random
import sqlite3
from pathlib import Path

MAX_VALUE_CHARS = 40


def _short(value: object) -> str:
    """Format one cell value for the prompt, cutting long text."""
    if value is None:
        return "NULL"
    text = str(value).replace("\n", " ")
    return text if len(text) <= MAX_VALUE_CHARS else text[: MAX_VALUE_CHARS - 3] + "..."


def _connect(db_path: Path) -> sqlite3.Connection:
    return sqlite3.connect(f"{Path(db_path).resolve().as_uri()}?mode=ro", uri=True)


def _tables_with_ddl(db_path: Path) -> list[tuple[str, str]]:
    con = _connect(db_path)
    try:
        return con.execute(
            "SELECT name, sql FROM sqlite_master "
            "WHERE type = 'table' AND name NOT LIKE 'sqlite_%' AND sql IS NOT NULL"
        ).fetchall()
    finally:
        con.close()


def table_names(db_path: Path, extra_dbs: list[Path] | None = None) -> list[str]:
    """List user tables in a SQLite database (and any extra databases).

    Args:
        db_path: Path to the ``.sqlite`` file.
        extra_dbs: Extra databases whose tables are listed after the main ones.

    Returns:
        Table names in creation order (internal ``sqlite_*`` tables excluded).
    """
    names = [n for n, _ in _tables_with_ddl(db_path)]
    for extra in extra_dbs or []:
        names += [n for n, _ in _tables_with_ddl(extra)]
    return names


def render_schema(
    db_path: Path,
    n_rows: int = 3,
    tables: list[str] | None = None,
    extra_dbs: list[Path] | None = None,
    shuffle_seed: int | None = None,
    notes: dict[str, dict[str, str]] | None = None,
) -> str:
    """Describe tables as CREATE TABLE statements, each followed by a few example rows.

    Args:
        db_path: Path to the ``.sqlite`` file.
        n_rows: Example rows per table (0 = none).
        tables: Only these tables (default: all).
        extra_dbs: Extra databases (e.g. distractor tables) rendered as if they were part of
            the main database.
        shuffle_seed: If given, shuffle the table order with this seed, so extra tables are
            mixed in with the real ones instead of all coming last.
        notes: Optional column notes ``{table: {column: note}}`` added after each table.

    Returns:
        Schema text for the prompt.
    """
    entries = [(name, ddl, Path(db_path)) for name, ddl in _tables_with_ddl(db_path)]
    for extra in extra_dbs or []:
        entries += [(name, ddl, Path(extra)) for name, ddl in _tables_with_ddl(extra)]
    if tables is not None:  # keep the caller's order (e.g. retrieval rank)
        by_name = {e[0]: e for e in entries}
        entries = [by_name[t] for t in tables if t in by_name]
    if shuffle_seed is not None:
        random.Random(shuffle_seed).shuffle(entries)

    parts = []
    cons: dict[Path, sqlite3.Connection] = {}
    try:
        for name, ddl, src in entries:
            block = ddl.strip()
            if n_rows > 0:
                con = cons.setdefault(src, _connect(src))
                quoted = '"' + name.replace('"', '""') + '"'
                cur = con.execute(f"SELECT * FROM {quoted} LIMIT ?", (n_rows,))
                cols = [d[0] for d in cur.description]
                rows = cur.fetchall()
                lines = ["\t".join(cols)] + ["\t".join(_short(v) for v in r) for r in rows]
                block += f"\n/* {len(rows)} example rows from {name}:\n" + "\n".join(lines) + "\n*/"
            if notes and notes.get(name):
                lines = [f"- {c}: {n}" for c, n in notes[name].items()]
                block += f"\n/* Column notes for {name}:\n" + "\n".join(lines) + "\n*/"
            parts.append(block)
    finally:
        for con in cons.values():
            con.close()
    return "\n\n".join(parts)

In [7]:
# --- copied from src/text2sql_eval/prompting.py in the companion repo ---
import logging
import re
import time
from dataclasses import dataclass


logger = logging.getLogger(__name__)

SYSTEM_PROMPT = (
    "You are an expert SQLite analyst. Given a database schema, an optional hint, and a "
    "question, write ONE SQLite query that answers the question.\n"
    "Rules:\n"
    "- Use only tables and columns that exist in the schema.\n"
    "- Quote identifiers that contain spaces or special characters with backticks.\n"
    "- Return exactly the columns the question asks for, in the order it asks for them.\n"
    "- Reply with the SQL inside a single ```sql code block and nothing else."
)


def build_messages(
    schema: str, question: str, evidence: str, values: str = ""
) -> list[dict[str, str]]:
    """Build chat messages for one question.

    Args:
        schema: Rendered schema text.
        question: Natural-language question.
        evidence: Expert hint (may be empty).
        values: Optional value-retrieval section (see ``values.format_hits``).

    Returns:
        A system and a user message.
    """
    hint = evidence.strip() or "(none)"
    found = f"{values}\n\n" if values else ""
    user = f"Schema:\n{schema}\n\nHint: {hint}\n\n{found}Question: {question}"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


_FENCED = re.compile(r"```(?:sql|sqlite)?\s*\n?(.*?)```", re.IGNORECASE | re.DOTALL)


def extract_sql(text: str) -> str:
    """Pull the SQL out of a model reply.

    Takes the last fenced code block if there is one, otherwise the whole reply.
    A trailing semicolon is removed.

    Args:
        text: Model reply.

    Returns:
        The SQL string (may be empty).
    """
    blocks = _FENCED.findall(text or "")
    sql = blocks[-1] if blocks else (text or "")
    return sql.strip().rstrip(";").strip()


def content_text(content: object) -> str:
    """Get the answer text from a message's content.

    Reasoning models (e.g. Mistral Large 4) return a list of chunks: ``ThinkChunk`` (the
    reasoning) and ``TextChunk`` (the answer). Only the text chunks are kept.

    Args:
        content: ``message.content``: a string, a list of chunks, or None.

    Returns:
        The answer text.
    """
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    return "".join(getattr(ch, "text", "") for ch in content if getattr(ch, "type", None) == "text")


@dataclass(frozen=True)
class Completion:
    """One model reply with token usage.

    Attributes:
        text: Reply content.
        prompt_tokens: Input tokens billed.
        completion_tokens: Output tokens billed.
        finish_reason: Why generation stopped (``"length"`` = hit max_tokens).
        texts: All sampled replies when ``n > 1`` (``text`` is the first one).
    """

    text: str
    prompt_tokens: int
    completion_tokens: int
    finish_reason: str = "stop"
    texts: tuple[str, ...] = ()


def chat(
    client: Mistral,
    model: str,
    messages: list[dict[str, str]],
    temperature: float = 0.0,
    max_tokens: int = 16384,  # Large 4 sometimes reasons past 8k tokens before answering
    retries: int = 6,
    n: int = 1,
) -> Completion:
    """Call Mistral chat completion with retry on transient errors.

    A rate-limit error (HTTP 429) waits a full minute, because limits are per minute.
    Other client errors (4xx) are raised at once. Anything else backs off 1, 2, 4, ... seconds.

    Args:
        client: A ``mistralai.client.Mistral`` instance.
        model: Pinned model name, e.g. ``ministral-8b-2512``.
        messages: Chat messages.
        temperature: Sampling temperature (0 for repeatable runs).
        max_tokens: Cap on output tokens.
        retries: Attempts before giving up.
        n: Number of replies to sample in one request (the prompt is billed once).

    Returns:
        The reply text(s) and token counts.

    Raises:
        RuntimeError: If every attempt failed.
    """
    last: Exception | None = None
    for attempt in range(retries):
        try:
            resp = client.chat.complete(
                model=model,
                messages=messages,
                temperature=temperature,
                top_p=1.0,  # some models (Large 4) require top_p=1 with temperature 0
                max_tokens=max_tokens,
                **({"n": n} if n > 1 else {}),
            )
            usage = resp.usage
            choice = resp.choices[0]
            finish = str(getattr(choice, "finish_reason", None) or "stop")
            if finish == "length":
                logger.warning("reply hit max_tokens=%d before finishing", max_tokens)
            texts = tuple(content_text(ch.message.content) for ch in resp.choices)
            return Completion(
                text=texts[0],
                prompt_tokens=usage.prompt_tokens or 0,
                completion_tokens=usage.completion_tokens or 0,
                finish_reason=finish,
                texts=texts,
            )
        except Exception as e:  # SDK raises several error types; retry transient ones
            status = getattr(e, "status_code", None)
            if status is not None and 400 <= status < 500 and status != 429:
                raise  # a bad request won't get better by retrying
            last = e
            wait = 60 if "429" in str(e) else 2**attempt
            logger.warning("chat call failed (%.80s); retry %d in %ds", e, attempt + 1, wait)
            time.sleep(wait)
    raise RuntimeError(f"chat call failed after {retries} attempts: {last}")

In [8]:
def evaluate(label: str, use_notes: bool = False, retry: bool = False) -> dict[int, bool]:
    """Answer every demo question, score it, and print the execution accuracy."""
    correct = {}
    for q in demo:
        db = db_file(q["db_id"])
        schema = render_schema(db, notes=NOTES.get(q["db_id"]) if use_notes else None)
        messages = build_messages(schema, q["question"], q["evidence"])
        if retry:
            pred = solve_with_retry(client, MODEL, messages, db, timeout_s=30).result
        else:
            pred = execute_sql(db, extract_sql(chat(client, MODEL, messages).text), timeout_s=30)
        correct[q["question_id"]] = compare(pred, GOLD[q["question_id"]], q["SQL"]).bird
    n = sum(correct.values())
    print(f"{label}: {n}/{len(demo)} correct ({n / len(demo):.0%})")
    return correct


NOTES: dict = {}
results = {"(a) plain schema": evaluate("(a) plain schema")}

(a) plain schema: 4/30 correct (13%)


## 4. Column notes: tell the model what the data means

Most failures we saw were **dirty data**, not bad SQL. BIRD ships a description file per table that already warns about some traps, e.g. for `superhero.weight_kg`: *"If the weight_kg is NULL or 0, it means the weight of the superhero is missing"*. Our plain prompt never showed it.

**Column notes** add, per column:
1. BIRD's description, and
2. an automatic **profile** that looks at the actual values and writes an instruction only when it finds a trap: many zeros, placeholders like `''` or `'-'`, numbers stored as text, numbers mixed with words, codes with leading zeros, comma-separated lists, and the full list of values for small columns.

In [9]:
# --- copied from src/text2sql_eval/notes.py in the companion repo ---
import csv
import io
import json
import logging
import re
import sqlite3
from collections import Counter
from pathlib import Path

logger = logging.getLogger(__name__)

SAMPLE_ROWS = 20_000
MAX_LISTED_VALUES = 12
PLACEHOLDERS = ("", "-", "--", "none", "null", "n/a", "na", "unknown", "?")
_NUMBER = re.compile(r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$")
_DESC_MAX = 160
_VALUE_DESC_MAX = 260


def _read_csv(path: Path) -> str:
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        return raw.decode("cp1252", errors="replace")


def _clean(text: str, limit: int) -> str:
    text = re.sub(r"(?i)commonsense evidence:\s*", "", text or "")
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 3].rstrip() + "..."


def read_descriptions(desc_dir: Path) -> dict[str, dict[str, str]]:
    """Read BIRD column descriptions.

    Args:
        desc_dir: A ``database_description`` folder (one CSV per table).

    Returns:
        ``{table_lower: {column_lower: text}}``; empty text is left out.
    """
    out: dict[str, dict[str, str]] = {}
    for p in sorted(Path(desc_dir).glob("*.csv")):
        cols: dict[str, str] = {}
        for row in csv.DictReader(io.StringIO(_read_csv(p))):
            row = {(k or "").strip(): (v or "").strip() for k, v in row.items()}
            name = row.get("original_column_name", "")
            desc = _clean(row.get("column_description", ""), _DESC_MAX)
            value = _clean(row.get("value_description", ""), _VALUE_DESC_MAX)
            text = desc + (f" Values: {value}" if value else "")
            if name and text.strip():
                cols[name.lower()] = text.strip()
        out[p.stem.lower()] = cols
    return out


def profile_column(values: list[object]) -> list[str]:
    """Describe data traps in a sample of column values, as short instructions.

    Args:
        values: Sampled values of one column (may include None).

    Returns:
        Zero or more short notes.
    """
    n = len(values)
    if n == 0:
        return []
    notes: list[str] = []
    non_null = [v for v in values if v is not None]
    if not non_null:
        return ["all sampled values are NULL"]

    numbers = [v for v in non_null if isinstance(v, (int, float))]
    texts = [v for v in non_null if isinstance(v, str)]

    zeros = sum(1 for v in numbers if v == 0)
    if numbers and zeros / n >= 0.05:
        notes.append(f"{zeros / n:.0%} of values are 0 (0 may mean missing; see description)")

    if texts:
        stripped = [t.strip() for t in texts]
        ph = Counter(s for s in stripped if s.lower() in PLACEHOLDERS)
        for val, c in ph.most_common(3):
            share = f"{c / n:.0%}" if c / n >= 0.01 else "<1%"
            notes.append(f"placeholder {val!r} in {share} of rows: treat as missing")
        real = [s for s in stripped if s.lower() not in PLACEHOLDERS]
        numeric = [s for s in real if _NUMBER.match(s)]
        if real and len(numeric) / len(real) >= 0.9:
            if any(len(s) > 1 and s.startswith("0") and s.isdigit() for s in numeric):
                notes.append("digit codes with leading zeros, stored as text: compare as text")
            else:
                notes.append("numbers stored as text: CAST(... AS REAL) before MIN/MAX/<,>")
        elif real and 0.2 <= len(numeric) / len(real) < 0.9:
            words = Counter(s for s in real if not _NUMBER.match(s)).most_common(3)
            shown = ", ".join(repr(w) for w, _ in words)
            notes.append(f"mixes numbers and words (e.g. {shown}): CAST only the numeric ones")
        listy = sum(1 for s in real if "," in s and len(s) <= 120)
        if real and listy / len(real) >= 0.05:
            notes.append("some values are comma-separated lists: match items with LIKE, not =")

    distinct = Counter(non_null)
    if 1 < len(distinct) <= MAX_LISTED_VALUES and len(non_null) >= 20:
        shown = ", ".join(repr(v) for v, _ in sorted(distinct.items(), key=lambda kv: -kv[1]))
        notes.append(f"all values: {shown}")
    return notes


def build_notes(db_path: Path, desc_dir: Path | None) -> dict[str, dict[str, str]]:
    """Build a note per column: BIRD description + profile instructions.

    Args:
        db_path: Path to the ``.sqlite`` file.
        desc_dir: BIRD ``database_description`` folder, or None.

    Returns:
        ``{table: {column: note}}`` for columns that have something to say.
    """
    descs = read_descriptions(desc_dir) if desc_dir and Path(desc_dir).is_dir() else {}
    con = sqlite3.connect(f"{Path(db_path).resolve().as_uri()}?mode=ro", uri=True)
    out: dict[str, dict[str, str]] = {}
    try:
        tables = [
            r[0]
            for r in con.execute(
                "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"
            )
        ]
        for t in tables:
            qt = '"' + t.replace('"', '""') + '"'
            cur = con.execute(f"SELECT * FROM {qt} LIMIT ?", (SAMPLE_ROWS,))
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
            tdesc = descs.get(t.lower(), {})
            notes_t: dict[str, str] = {}
            for i, c in enumerate(cols):
                parts = []
                if tdesc.get(c.lower()):
                    parts.append(tdesc[c.lower()])
                prof = profile_column([r[i] for r in rows])
                if prof:
                    parts.append("[data] " + "; ".join(prof))
                if parts:
                    notes_t[c] = " ".join(parts)
            if notes_t:
                out[t] = notes_t
    finally:
        con.close()
    return out


def format_notes(table: str, notes: dict[str, str]) -> str:
    """Render one table's notes as a SQL comment block for the prompt."""
    if not notes:
        return ""
    lines = [f"- {c}: {n}" for c, n in notes.items()]
    return f"/* Column notes for {table}:\n" + "\n".join(lines) + "\n*/"


def save_notes(notes: dict[str, dict[str, str]], path: Path) -> None:
    """Write notes to JSON."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(notes, indent=1, ensure_ascii=False), encoding="utf-8")


def load_notes(path: Path) -> dict[str, dict[str, str]]:
    """Read notes written by save_notes."""
    return json.loads(Path(path).read_text(encoding="utf-8"))

In [10]:
NOTES = {db: build_notes(db_file(db), DB_ROOT / db / "database_description") for db in DEMO_DBS}
print("note for superhero.weight_kg:\n ", NOTES["superhero"]["superhero"]["weight_kg"])
print("note for Laboratory.SSA:\n ", NOTES["thrombosis_prediction"]["Laboratory"]["SSA"])

note for superhero.weight_kg:
  the weight of the superhero Values: The unit of weight is kilogram. If the weight_kg is NULL or 0, it means the weight of the superhero is missing. [data] 23% of values are 0 (0 may mean missing; see description)
note for Laboratory.SSA:
  anti-SSA Values: Normal range: N= -, +- [data] mixes numbers and words (e.g. 'negative'): CAST only the numeric ones; all values: '0', 'negative', '16', '64', '4', '1', '256'


In [11]:
results["(a) + column notes"] = evaluate("(a) + column notes", use_notes=True)

(a) + column notes: 13/30 correct (43%)


## 5. Execute and retry

The model writes SQL, we run it, and if it fails (an error, a timeout, or zero rows) we send that feedback back and ask for a fix, up to 3 attempts. This fixes SQL that doesn't run; it rarely fixes SQL that runs but answers the wrong question.

In [12]:
# --- copied from src/text2sql_eval/agent.py in the companion repo ---
from dataclasses import dataclass, field
from pathlib import Path


NO_SQL = "no SQL found in the reply"
NO_SQL_FEEDBACK = (
    "Your reply did not contain a SQL query (it may have run out of length while reasoning). "
    "Reason briefly, then reply with one SQLite query in a ```sql code block."
)


def feedback_for(result: ExecResult) -> str | None:
    """Turn an execution result into feedback for the model, or None if it looks fine.

    Args:
        result: Result of running the model's SQL.

    Returns:
        A short message describing the problem, or None if the query returned rows.
    """
    if result.timed_out:
        return (
            "The query did not finish within the time limit. It is probably missing a join "
            "condition or scanning too much. Rewrite it to be correct and efficient."
        )
    if result.error is not None:
        return f"The query failed with this SQLite error: {result.error}\nFix the query."
    if result.rows is not None and len(result.rows) == 0:
        return (
            "The query ran but returned 0 rows. Check the filter values (exact spelling and "
            "format in the example rows), the join conditions, and the tables used. If you are "
            "sure 0 rows is correct, repeat the same query."
        )
    return None


@dataclass
class AgentResult:
    """Outcome of the execute-and-retry loop.

    Attributes:
        sql: Final SQL.
        result: Execution result of the final SQL.
        attempts: Number of model calls made.
        first_sql: SQL from the first attempt.
        first_result: Execution result of the first attempt.
        feedback: Feedback messages sent, in order.
        prompt_tokens: Total input tokens over all calls.
        completion_tokens: Total output tokens over all calls.
    """

    sql: str
    result: ExecResult
    attempts: int
    first_sql: str
    first_result: ExecResult
    feedback: list[str] = field(default_factory=list)
    prompt_tokens: int = 0
    completion_tokens: int = 0


def solve_with_retry(
    client: Mistral,
    model: str,
    messages: list[dict[str, str]],
    db_path: Path,
    timeout_s: float,
    max_attempts: int = 3,
    attach: list[Path] | None = None,
) -> AgentResult:
    """Ask for SQL, run it, and retry with execution feedback until it looks fine.

    Args:
        client: Mistral client.
        model: Pinned model name.
        messages: Initial system + user messages (same as approach a).
        db_path: SQLite database to run against.
        timeout_s: Timeout per execution.
        max_attempts: Maximum model calls (1 = no retry).
        attach: Extra databases to attach when running SQL (e.g. distractor tables).

    Returns:
        The final SQL and result, plus the first attempt and token totals.
    """
    history = list(messages)
    feedback: list[str] = []
    tok_in = tok_out = 0
    first_sql = ""
    first_result: ExecResult | None = None
    sql = ""
    result = ExecResult(rows=None, error="no attempt")
    for attempt in range(1, max_attempts + 1):
        reply = chat(client, model, history)
        tok_in += reply.prompt_tokens
        tok_out += reply.completion_tokens
        sql = extract_sql(reply.text)
        result = (
            execute_sql(db_path, sql, timeout_s=timeout_s, attach=attach)
            if sql
            else ExecResult(rows=None, error=NO_SQL)
        )
        if first_result is None:
            first_sql, first_result = sql, result
        if not sql:
            note: str | None = NO_SQL_FEEDBACK
        else:
            note = feedback_for(result)
        if note is None or attempt == max_attempts:
            return AgentResult(
                sql=sql,
                result=result,
                attempts=attempt,
                first_sql=first_sql,
                first_result=first_result,
                feedback=feedback,
                prompt_tokens=tok_in,
                completion_tokens=tok_out,
            )
        feedback.append(note)
        history += [
            # the API rejects an empty assistant message (e.g. reply was all reasoning)
            {"role": "assistant", "content": reply.text or "(no answer)"},
            {"role": "user", "content": note},
        ]
    raise AssertionError("unreachable")  # pragma: no cover

In [13]:
results["(c) retry + column notes"] = evaluate("(c) retry + column notes", use_notes=True, retry=True)

(c) retry + column notes: 13/30 correct (43%)


## 6. Results

**Demo (30 questions).** With only 30 questions, a single question is 3 points, and the same prompt can give different SQL on a rerun even at temperature 0, so treat these numbers as an illustration.

In [14]:
for label, res in results.items():
    print(f"{label:28s} {sum(res.values()):2d}/{len(res)}")
base, best = results["(a) plain schema"], results["(c) retry + column notes"]
print("\nwrong -> right:", [q for q in base if best[q] and not base[q]])
print("right -> wrong:", [q for q in base if base[q] and not best[q]])

(a) plain schema              4/30
(a) + column notes           13/30
(c) retry + column notes     13/30

wrong -> right: [10, 339, 348, 350, 356, 363, 374, 375, 414, 433, 438]
right -> wrong: [387, 413]


**Full runs (all 500 questions, BIRD rule).** These come from the companion repo, where every run is reproducible with one command and every per-question answer is saved. Large 4 reasons before answering and costs about 15x more per question than Small 4, so some of its extras were run on a fixed 200-question subset.

| Approach | Ministral 3 8B (500 q) | Mistral Small 4 (500 q) | Mistral Large 4 (500 q) | Mistral Large 4 (200 q subset) |
|---|---|---|---|---|
| (a) full schema | 10.6% | 18.6% | 60.2% | 61.5% |
| (b) schema retrieval | 10.2% | 20.4% | 60.8% | 61.5% |
| (c) execute and retry | 14.0% | 20.6% | 65.0% | 66.0% |
| (a) + value retrieval | 10.2% | 18.4% | – | 60.5% |
| majority vote, 5 samples | 10.6% | 21.0% | – | – |
| **(a) + column notes** | **16.4%** | **30.8%** | – | **72.0%** |
| **(c) + column notes** | **21.0%** | **32.8%** | – | – |

What we learned:
- **Column notes are the biggest win** for every model (+5.8 points for Ministral 3 8B, +12.2 for Small 4, +10.5 for Large 4 on the 200-question subset; all statistically significant in a paired McNemar test). They help even though BIRD's per-question hints are already in the prompt: the hints explain the *question*, the notes explain the *data*.
- **Execute-and-retry** fixes SQL that fails to run, and stacks with column notes (Ministral 3 8B: 10.6% → 21.0%).
- **Value retrieval** (putting stored values that match words in the question into the prompt) and **majority voting** gave no clear gain here; for Large 4, the 5 voted samples agreed on most questions.
- **Schema retrieval** (showing only relevant tables) changes nothing on BIRD's small databases, but it protects accuracy when the database is large. With 30 unrelated "distractor" tables added, the plain prompt loses accuracy and retrieval does not:

| Model | (a) without → with 30 distractor tables | (b) without → with |
|---|---|---|
| Ministral 3 8B | 10.6% → 7.8% | 10.2% → 10.4% |
| Mistral Small 4 | 18.6% → 13.8% | 20.4% → 18.8% |
| Mistral Large 4 (200 q) | 61.5% → 56.0% | 61.5% → 64.0% |

## 7. Why do the remaining answers fail?

We read 50 random failures of Mistral Large 4 with execute-and-retry (before column notes):

| Error type | Count (of 50) |
|---|---|
| Dirty data (0 or '-' meaning missing, numbers stored as text, lists in one field) | 21 |
| Wrong filter (question misread, extra condition added) | 9 |
| Wrong column | 6 |
| Wrong filter value ('-' instead of the stored 'negative') | 4 |
| Wrong aggregation (COUNT instead of COUNT DISTINCT) | 4 |
| Format only (answer right, e.g. 2015 vs '2015') | 4 |
| Wrong join | 1 |
| Ambiguous question | 1 |

One real example per type, with the model's SQL and the gold SQL, is in [docs/error_examples.md](https://github.com/smasoudrezvani/mistral-text2sql-cookbook/blob/main/docs/error_examples.md). Dirty data was the largest group, which is what led us to column notes.

## Takeaways

1. Measure with **execution accuracy** on real, messy data; a demo database hides these problems.
2. Before adding agents or more model calls, **give the model the data's meaning**: column descriptions plus a cheap automatic profile.
3. Add an **execute-and-retry** loop for SQL that doesn't run.
4. Use **schema retrieval** only when the schema is large.

## References

- BIRD: Li et al., 2023, [arXiv 2305.03111](https://arxiv.org/abs/2305.03111)
- Synthetic column descriptions: Wretblad et al., 2024, [arXiv 2408.04691](https://arxiv.org/abs/2408.04691)
- Automatic metadata extraction (profiling): Shkapenyuk et al., 2025, [arXiv 2505.19988](https://arxiv.org/abs/2505.19988)
- CHESS (value and context retrieval): Talaei et al., 2024, [arXiv 2405.16755](https://arxiv.org/abs/2405.16755)
- Self-consistency (majority voting): Wang et al., 2022, [arXiv 2203.11171](https://arxiv.org/abs/2203.11171)